### 1. Series = 1D NumPy Array + Index

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
prices = np.array([150.0, 152.3, 151.8, 153.2])

In [ ]:
prices = pd.Series([150.0, 152.3, 151.8, 153.2], index=['2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05'])

print(prices['2024-01-02']) # You can use the index as a reference to the prices series
#index act as metadata

#important concepts here:
"""
• use of pd.Series([dataset], index=[labels])
• prices[label element]
"""


150.0


'\n• use of pd.Series([dataset], index=[labels])\n\n• prices[label element]\n'

### 2. DataFrame = 2D NumPY Array + Row Index + Column Index

In [ ]:
#assuming you have your portfolio array
# Numpy approach
portfolio = np.array([
    [150.0, 2800.0, 180.0],  # day 0: AAPL, TSLA, MSFT
    [152.3, 2750.0, 182.1],  # day 1
    [151.8, 2820.0, 181.5],  # day 2
    [153.2, 2790.0, 183.0]   # day 3
])


# Pandas approach
portfolio = pd.DataFrame(
    [[150.0, 2800.0, 180.0],  # day 0: AAPL, TSLA, MSFT
     [152.3, 2750.0, 182.1],  # day 1
     [151.8, 2820.0, 181.5],  # day 2
     [153.2, 2790.0, 183.0]],   # day 3
     index=['2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05'],
     columns=['AAPL','TSLA','MSFT'] 
    )
    # NOTE: The length of each of the index/columns must match the dimensions of the 2D Array! Try removing an element from each, you should get an error


print(portfolio.loc['2024-01-03','MSFT'])
# .loc[index, columns]
#print(portfolio.loc['MSFT','2024-01-03']) <- this shouldn't work !!





182.1


In [23]:
# Q: What is the difference between pd.DataFrame() and pd.Series()?

# NumPy equivalents:
np_array_1d = np.array([150.0, 152.3, 151.8])      # 1D array
np_array_2d = np.array([[150.0, 2800.0, 180.0],    # 2D array
                        [152.3, 2750.0, 182.1],
                        [151.8, 2820.0, 181.5]])

# Pandas equivalents (NumPy + labels):
series = pd.Series([150.0, 152.3, 151.8], 
                   index=['2024-01-02', '2024-01-03', '2024-01-04'],
                   name='AAPL')  # ONE column with ONE label

dataframe = pd.DataFrame({
    'AAPL': [150.0, 152.3, 151.8],
    'TSLA': [2800.0, 2750.0, 2820.0],
    'MSFT': [180.0, 182.1, 181.5]
}, index=['2024-01-02', '2024-01-03', '2024-01-04'])
# MULTIPLE columns, each with its OWN label

### 3. Automatic Alignment

In [ ]:
# AAPL traded all 4 days
aapl = pd.Series([0.01, 0.02, -0.01, 0.015], 
                 index=['2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05'])

# TSLA was halted on Jan 3rd (missing data)
tsla = pd.Series([0.015, -0.008, 0.012], 
                 index=['2024-01-02', '2024-01-04', '2024-01-05'])

# What happens when you compute the spread?
spread = aapl - tsla

print(spread) # you can see in the output that nan's will be outputted.
# so you could use NumPy's to identify them


2024-01-02   -0.005
2024-01-03      NaN
2024-01-04   -0.002
2024-01-05    0.003
dtype: float64


### On making sure arrays are aligned before .to_numpy()

**Short answer:** No, "aligned" ≠ "remove NaNs". It means something more fundamental.

---

## What "Aligned" Actually Means

**Aligned = same index in the same order**

When you convert to NumPy, you lose the labels. So position 0 in one array must correspond to the **same logical entity** as position 0 in another array.

---

## Case 1: Aligned Data (NaNs are fine!) ✅

```python
# Both Series have the SAME index
aapl = pd.Series([150.0, 152.3, np.nan, 153.2], 
                 index=['2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05'])

tsla = pd.Series([2800.0, np.nan, 2820.0, 2790.0], 
                 index=['2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05'])

# These ARE aligned because:
# - Same index labels
# - Same order
# - Same length (4)

# Converting to NumPy is SAFE:
aapl_np = aapl.to_numpy()  # [150.0, 152.3, nan, 153.2]
tsla_np = tsla.to_numpy()  # [2800.0, nan, 2820.0, 2790.0]

# Operations work correctly:
spread = aapl_np - tsla_np
# [-2650.0, nan, nan, -2636.8]

# Why it's correct:
# Position 0: 150.0 - 2800.0 = -2650.0  (both from 2024-01-02) ✅
# Position 1: 152.3 - nan = nan         (both from 2024-01-03) ✅
# Position 2: nan - 2820.0 = nan        (both from 2024-01-04) ✅
# Position 3: 153.2 - 2790.0 = -2636.8  (both from 2024-01-05) ✅
```

**The NaNs are in the RIGHT positions** — they represent missing data on the correct dates.

---

## Case 2: NOT Aligned (Dangerous!) ⚠️

```python
# Different indexes - DANGER ZONE
aapl = pd.Series([150.0, 152.3, 151.8], 
                 index=['2024-01-02', '2024-01-03', '2024-01-04'])

tsla = pd.Series([2800.0, 2820.0, 2790.0], 
                 index=['2024-01-02', '2024-01-04', '2024-01-05'])
                 # ⚠️ Missing Jan 3, has Jan 5 instead!

# Naively converting to NumPy:
aapl_np = aapl.to_numpy()  # [150.0, 152.3, 151.8]
tsla_np = tsla.to_numpy()  # [2800.0, 2820.0, 2790.0]

# This subtraction is WRONG:
spread = aapl_np - tsla_np
# [-2650.0, -2667.7, -2638.2]

# WHY IT'S WRONG:
# Position 0: 150.0 - 2800.0 ✅ (both Jan 2)
# Position 1: 152.3 - 2820.0 ❌ (Jan 3 - Jan 4) DIFFERENT DATES!
# Position 2: 151.8 - 2790.0 ❌ (Jan 4 - Jan 5) DIFFERENT DATES!
```

You just computed the spread between prices on **different dates**. Garbage result!

---

## How to Check Alignment Before Converting

```python
# Method 1: Check if indexes are identical
aapl.index.equals(tsla.index)  # → True or False

# Method 2: Visual inspection (small datasets)
print(aapl.index)
print(tsla.index)

# Method 3: Check shapes
print(aapl.shape, tsla.shape)  # Must be the same

# Method 4: The safest approach - combine into DataFrame FIRST
df = pd.DataFrame({'AAPL': aapl, 'TSLA': tsla})
# Pandas will align them automatically (outer join)
# Then inspect:
print(df.isna().sum())  # See where NaNs appeared
```

---

## When You Actually Need to Handle NaNs

**Scenario A: Your NumPy operation can't handle NaNs**

```python
# Example: np.corrcoef doesn't handle NaNs well
returns_df = pd.DataFrame({'AAPL': aapl_returns, 'TSLA': tsla_returns})

# Option 1: Drop rows with ANY NaN
clean_df = returns_df.dropna()
corr = np.corrcoef(clean_df['AAPL'].to_numpy(), 
                   clean_df['TSLA'].to_numpy())

# Option 2: Use Pandas (which handles NaNs properly)
corr = returns_df.corr()  # Better! Uses pairwise-complete observations
```

**Scenario B: You want matching observations only**

```python
# You have misaligned data and want ONLY overlapping dates
aapl = pd.Series([150.0, 152.3, 151.8, 153.2], 
                 index=['2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05'])

tsla = pd.Series([2800.0, 2820.0, 2790.0], 
                 index=['2024-01-02', '2024-01-04', '2024-01-05'])

# Step 1: Combine into DataFrame (aligns automatically)
df = pd.DataFrame({'AAPL': aapl, 'TSLA': tsla})
#               AAPL    TSLA
# 2024-01-02  150.0  2800.0
# 2024-01-03  152.3     NaN   ← TSLA missing
# 2024-01-04  151.8  2820.0
# 2024-01-05  153.2  2790.0

# Step 2: Drop rows with NaNs
df_clean = df.dropna()
#               AAPL    TSLA
# 2024-01-02  150.0  2800.0
# 2024-01-04  151.8  2820.0
# 2024-01-05  153.2  2790.0

# Step 3: NOW it's safe to convert to NumPy
aapl_clean = df_clean['AAPL'].to_numpy()  # [150.0, 151.8, 153.2]
tsla_clean = df_clean['TSLA'].to_numpy()  # [2800.0, 2820.0, 2790.0]

# Now they're truly aligned (position 0 = Jan 2, position 1 = Jan 4, etc.)
```

---

## The Key Principle 🔑

**"Aligned" means:**
1. ✅ Same index (same labels, same order)
2. ✅ Same length
3. ✅ Position `i` in both arrays refers to the same logical observation

**"Aligned" does NOT mean:**
- ❌ No NaNs
- ❌ No missing data
- ❌ All values filled

---

## Practical Workflow for Real Quant Work

```python
# You have misaligned stock data
prices = {
    'AAPL': aapl_series,  # Different indexes
    'TSLA': tsla_series,
    'MSFT': msft_series
}

# Step 1: Combine into DataFrame (Pandas handles alignment)
df = pd.DataFrame(prices)

# Step 2: Decide on your NaN strategy
# Option A: Keep NaNs (many operations handle them)
returns = df.pct_change()  # Works fine with NaNs

# Option B: Drop NaNs (need complete cases only)
df_clean = df.dropna()
returns = df_clean.pct_change()

# Option C: Forward-fill (assume price doesn't change)
df_filled = df.ffill()  # Fill missing dates with last known price
returns = df_filled.pct_change()

# Step 3: NOW convert to NumPy if needed (data is aligned)
returns_np = returns.to_numpy()  # Safe because df ensured alignment
```

---

## Bottom Line

**"Aligned" is about matching indexes, not about removing NaNs.**

You need to remove NaNs if:
- Your downstream NumPy operation can't handle them
- You want only observations where ALL stocks have data

You DON'T need to remove NaNs if:
- Your operation handles NaNs correctly
- The NaNs are already in the right positions (representing true missing data)

Does this clarify the distinction? The key insight: **Pandas aligns on labels, NumPy operates on positions**. So before dropping labels (converting to NumPy), make sure positions already mean the right thing! 🎯